In [3]:
import pandas as pd
import numpy as np

In [4]:
# Load the dataset

df = pd.read_csv(
    "AB_NYC_2019.csv",
    dtype={
        "id": "int64",
        "name": "string",
        "host_id": "int64",
        "host_name": "string",
        "neighbourhood_group": "string",
        "neighbourhood": "string",
        "latitude": "float64",
        "longitude": "float64",
        "room_type": "string",
        "price": "float64",
        "minimum_nights": "int64",
        "number_of_reviews": "int64",
        "last_review": "string",
        "reviews_per_month": "float64",
        "calculated_host_listings_count": "int64",
        "availability_365": "int64"
    },
    parse_dates=["last_review"]
)

print("Shape:", df.shape)
print("\nMemory usage before optimization:")
df.info(memory_usage="deep")

Shape: (48895, 16)

Memory usage before optimization:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48895 entries, 0 to 48894
Data columns (total 16 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   id                              48895 non-null  int64  
 1   name                            48879 non-null  string 
 2   host_id                         48895 non-null  int64  
 3   host_name                       48874 non-null  string 
 4   neighbourhood_group             48895 non-null  string 
 5   neighbourhood                   48895 non-null  string 
 6   latitude                        48895 non-null  float64
 7   longitude                       48895 non-null  float64
 8   room_type                       48895 non-null  string 
 9   price                           48895 non-null  float64
 10  minimum_nights                  48895 non-null  int64  
 11  number_of_reviews               48895 n

In [5]:
memory_before = df.memory_usage(deep=True).sum()

print("Memory before optimization:",
      memory_before, "bytes")
print("Memory before optimization:",
      round(memory_before / (1024 ** 2), 2), "MB")

Memory before optimization: 22526028 bytes
Memory before optimization: 21.48 MB


In [6]:
# Downcast numeric columns

numeric_columns = df.select_dtypes(
    include=["int64", "float64"]
).columns

for column in numeric_columns:
    if pd.api.types.is_integer_dtype(df[column]):
        df[column] = pd.to_numeric(
            df[column],
            downcast="integer"
        )
    else:
        df[column] = pd.to_numeric(
            df[column],
            downcast="float"
        )

# Convert low-cardinality text columns to category

text_columns = df.select_dtypes(
    include=["string", "object"]
).columns

for column in text_columns:
    if df[column].nunique(dropna=False) / len(df) < 0.5:
        df[column] = df[column].astype("category")

In [7]:
memory_after = df.memory_usage(deep=True).sum()

saving_percent = (
    (memory_before - memory_after)
    / memory_before
) * 100

print("Memory after optimization:",
      memory_after, "bytes")

print("Memory after optimization:",
      round(memory_after / (1024 ** 2), 2), "MB")

print("Memory saved:",
      round(memory_before - memory_after, 2), "bytes")

print("Memory saving percentage:",
      round(saving_percent, 2), "%")

Memory after optimization: 7315771 bytes
Memory after optimization: 6.98 MB
Memory saved: 15210257 bytes
Memory saving percentage: 67.52 %


In [8]:
# Q2 - Data Profiling

profile = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2),
    "unique_count": df.nunique(dropna=False),
    "most_frequent_value": df.mode(dropna=True).iloc[0],
    "frequency": df.apply(
        lambda column: column.value_counts(dropna=True).iloc[0]
        if column.dropna().shape[0] > 0 else 0
    )
})

profile = profile.reset_index()
profile = profile.rename(columns={"index": "column"})

display(profile)

,column,dtype,missing_count,missing_percent,unique_count,most_frequent_value,frequency
0,id,int32,0,0.00,48895,2539,1
1,name,string,16,0.03,47906,Hillside Hotel,18
2,host_id,int32,0,0.00,37457,219517861.0,327
3,host_name,category,21,0.04,11453,Michael,417
4,neighbourhood_group,category,0,0.00,5,Manhattan,21661
5,neighbourhood,category,0,0.00,221,Williamsburg,3920
6,latitude,float32,0,0.00,19048,40.718128,18
7,longitude,float32,0,0.00,14718,-73.956772,18
8,room_type,category,0,0.00,3,Entire home/apt,25409
9,price,float32,0,0.00,674,100.0,2051


In [9]:
## 3 Exact duplicate rows
exact_duplicates = df.duplicated().sum()

print("Exact duplicate rows:", exact_duplicates)


# Duplicate records based on key
key_columns = ["id"]

key_duplicates = df.duplicated(
    subset=key_columns
).sum()

print(
    "Duplicate rows based on key columns",
    key_columns,
    ":",
    key_duplicates
)

Exact duplicate rows: 0
Duplicate rows based on key columns ['id'] : 0


In [10]:
##4
print("Value counts BEFORE cleaning:")
print(df["room_type"].value_counts(dropna=False))


df["room_type"] = (
    df["room_type"]
    .astype("string")
    .str.strip()
    .str.lower()
)


df["room_type"] = df["room_type"].replace({
    "entire home/apt": "Entire home/apt",
    "private room": "Private room",
    "shared room": "Shared room"
})


print("\nValue counts AFTER cleaning:")
print(df["room_type"].value_counts(dropna=False))

Value counts BEFORE cleaning:
room_type
Entire home/apt    25409
Private room       22326
Shared room         1160
Name: count, dtype: int64

Value counts AFTER cleaning:
room_type
Entire home/apt    25409
Private room       22326
Shared room         1160
Name: count, dtype: Int64


In [11]:
## 5
df["number_from_name"] = (
    df["name"]
    .astype("string")
    .str.extract(
        r"(\d+(?:\.\d+)?)",
        expand=False
    )
)

df["number_from_name"] = pd.to_numeric(
    df["number_from_name"],
    errors="coerce"
)

not_matched = df["number_from_name"].isna().sum()

print("Rows where number was not matched:",
      not_matched)

display(
    df[
        ["name", "number_from_name"]
    ].head(20)
)

Rows where number was not matched: 33056


,name,number_from_name
0,Clean & quiet apt home by the park,<NA>
1,Skylit Midtown Castle,<NA>
2,THE VILLAGE OF HARLEM....NEW YORK !,<NA>
3,Cozy Entire Floor of Brownstone,<NA>
4,Entire Apt: Spacious Studio/Loft by central park,<NA>
5,Large Cozy 1 BR Apartment In Midtown East,1.0
6,BlissArtsSpace!,<NA>
7,Large Furnished Room Near B'way,<NA>
8,Cozy Clean Guest Room - Family Apt,<NA>
9,Cute & Cozy Lower East Side 1 bdrm,1.0


In [12]:
## 6
df["last_review"] = pd.to_datetime(
    df["last_review"],
    errors="coerce"
)

failed_dates = df["last_review"].isna().sum()

print("Failed/missing date values:",
      failed_dates)


df["review_year"] = df["last_review"].dt.year

df["review_month"] = df["last_review"].dt.month

df["review_day_of_week"] = (
    df["last_review"].dt.day_name()
)

df["review_quarter"] = (
    df["last_review"].dt.quarter
)


display(
    df[
        [
            "last_review",
            "review_year",
            "review_month",
            "review_day_of_week",
            "review_quarter"
        ]
    ].head(10)
)

Failed/missing date values: 10052


,last_review,review_year,review_month,review_day_of_week,review_quarter
0,2018-10-19,2018.0,10.0,Friday,4.0
1,2019-05-21,2019.0,5.0,Tuesday,2.0
2,NaT,NaN,NaN,NaN,NaN
3,2019-07-05,2019.0,7.0,Friday,3.0
4,2018-11-19,2018.0,11.0,Monday,4.0
5,2019-06-22,2019.0,6.0,Saturday,2.0
6,2017-10-05,2017.0,10.0,Thursday,4.0
7,2019-06-24,2019.0,6.0,Monday,2.0
8,2017-07-21,2017.0,7.0,Friday,3.0
9,2019-06-09,2019.0,6.0,Sunday,2.0


In [13]:
## 7
def price_range(series):
    return series.max() - series.min()


group_summary = (
    df.groupby("neighbourhood_group")
      .agg(
          listing_count=("id", "count"),

          average_price=("price", "mean"),

          median_price=("price", "median"),

          average_reviews=("number_of_reviews", "mean"),

          price_range=("price", price_range)
      )
      .reset_index()
)


display(group_summary)

/tmp/ipykernel_1038/1010758131.py:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("neighbourhood_group")


,neighbourhood_group,listing_count,average_price,median_price,average_reviews,price_range
0,Bronx,1091,87.496796,65.0,26.004583,2500.0
1,Brooklyn,20104,124.383209,90.0,24.202845,10000.0
2,Manhattan,21661,196.875809,150.0,20.985596,10000.0
3,Queens,5666,99.517647,75.0,27.700318,9990.0
4,Staten Island,373,114.812332,75.0,30.941019,4987.0


In [14]:
## 8
group_mean_price = (
    df.groupby("neighbourhood_group")["price"]
      .transform("mean")
)

df["price_minus_group_mean"] = (
    df["price"] - group_mean_price
)


display(
    df[
        [
            "neighbourhood_group",
            "price",
            "price_minus_group_mean"
        ]
    ].head(10)
)

/tmp/ipykernel_1038/749955605.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("neighbourhood_group")["price"]


,neighbourhood_group,price,price_minus_group_mean
0,Brooklyn,149.0,24.616791
1,Manhattan,225.0,28.124191
2,Manhattan,150.0,-46.875809
3,Brooklyn,89.0,-35.383209
4,Manhattan,80.0,-116.875809
5,Manhattan,200.0,3.124191
6,Brooklyn,60.0,-64.383209
7,Manhattan,79.0,-117.875809
8,Manhattan,79.0,-117.875809
9,Manhattan,150.0,-46.875809


In [15]:
## 9
pivot_source = (
    df.groupby(
        ["neighbourhood_group", "room_type"],
        observed=True
    )["price"]
    .mean()
    .reset_index(name="avg_price")
)


pivot_table = pivot_source.pivot(
    index="neighbourhood_group",
    columns="room_type",
    values="avg_price"
)


print("PIVOT TABLE:")
display(pivot_table)

PIVOT TABLE:


room_type,Entire home/apt,Private room,Shared room
neighbourhood_group,,,
Bronx,127.506599,66.788345,59.799999
Brooklyn,178.327545,76.500099,50.527843
Manhattan,249.239105,116.776619,88.977081
Queens,147.050568,71.762459,69.020203
Staten Island,173.846588,62.292553,57.444443


In [16]:
melted = (
    pivot_table
    .reset_index()
    .melt(
        id_vars="neighbourhood_group",
        var_name="room_type",
        value_name="avg_price"
    )
    .dropna()
)


print("MELTED DATA:")
display(melted)

MELTED DATA:


,neighbourhood_group,room_type,avg_price
0,Bronx,Entire home/apt,127.506599
1,Brooklyn,Entire home/apt,178.327545
2,Manhattan,Entire home/apt,249.239105
3,Queens,Entire home/apt,147.050568
4,Staten Island,Entire home/apt,173.846588
5,Bronx,Private room,66.788345
6,Brooklyn,Private room,76.500099
7,Manhattan,Private room,116.776619
8,Queens,Private room,71.762459
9,Staten Island,Private room,62.292553


In [17]:
original_sorted = (
    pivot_source
    .sort_values(
        ["neighbourhood_group", "room_type"]
    )
    .reset_index(drop=True)
)


melted_sorted = (
    melted
    .sort_values(
        ["neighbourhood_group", "room_type"]
    )
    .reset_index(drop=True)
)


same_groups = (
    original_sorted[
        ["neighbourhood_group", "room_type"]
    ].equals(
        melted_sorted[
            ["neighbourhood_group", "room_type"]
        ]
    )
)


same_values = np.allclose(
    original_sorted["avg_price"].values,
    melted_sorted["avg_price"].values,
    equal_nan=True
)


print("Groups identical:", same_groups)
print("Values identical:", same_values)
print("Round-trip lossless:",
      same_groups and same_values)

Groups identical: True
Values identical: True
Round-trip lossless: True


In [18]:
left_df = df[
    [
        "id",
        "host_id",
        "price"
    ]
].copy()


right_df = df[
    [
        "host_id",
        "room_type",
        "availability_365"
    ]
].copy()

In [19]:
inner_merge = left_df.merge(
    right_df,
    on="host_id",
    how="inner"
)

print(
    "Inner merge row count:",
    len(inner_merge)
)

Inner merge row count: 349305


In [20]:
left_merge = left_df.merge(
    right_df,
    on="host_id",
    how="left"
)

print(
    "Left merge row count:",
    len(left_merge)
)

Left merge row count: 349305


In [21]:
outer_merge = left_df.merge(
    right_df,
    on="host_id",
    how="outer"
)

print(
    "Outer merge row count:",
    len(outer_merge)
)

Outer merge row count: 349305


In [23]:
##11
df_sorted = df.sort_values(
    [
        "neighbourhood_group",
        "last_review"
    ]
).copy()

In [24]:
df_sorted = df.sort_values(
    [
        "neighbourhood_group",
        "last_review"
    ]
).copy()

In [25]:
df_sorted["price_rank_within_group"] = (
    df_sorted
    .groupby("neighbourhood_group")["price"]
    .rank(
        method="dense",
        ascending=False
    )
)

/tmp/ipykernel_1038/3843024793.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("neighbourhood_group")["price"]


In [26]:
df_sorted["price_rank_within_group"] = (
    df_sorted
    .groupby("neighbourhood_group")["price"]
    .rank(
        method="dense",
        ascending=False
    )
)

/tmp/ipykernel_1038/3843024793.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("neighbourhood_group")["price"]


In [27]:
## 12
price_limit = df["price"].quantile(0.99)
minimum_nights_limit = df["minimum_nights"].quantile(0.99)

suspicious_rows = df[
    (df["price"] == 0) |
    (df["price"] > price_limit) |
    (df["minimum_nights"] > minimum_nights_limit)
].head(3)


display(
    suspicious_rows[
        [
            "id",
            "name",
            "price",
            "minimum_nights",
            "number_of_reviews",
            "room_type",
            "neighbourhood_group"
        ]
    ]
)

,id,name,price,minimum_nights,number_of_reviews,room_type,neighbourhood_group
14,6090,West Village Nest - Superhost,120.0,90,27,Entire home/apt,Manhattan
36,11452,Clean and Quiet in Brooklyn,35.0,60,0,Private room,Brooklyn
61,15396,Sunny & Spacious Chelsea Apartment,375.0,180,5,Entire home/apt,Manhattan


In [28]:
## 13
!pip install pyarrow -q

In [29]:
csv_path = "cleaned_airbnb.csv"
parquet_path = "cleaned_airbnb.parquet"


# Save CSV
df.to_csv(
    csv_path,
    index=False
)


# Save Parquet
df.to_parquet(
    parquet_path,
    index=False
)

In [30]:
csv_path = "cleaned_airbnb.csv"
parquet_path = "cleaned_airbnb.parquet"


# Save CSV
df.to_csv(
    csv_path,
    index=False
)


# Save Parquet
df.to_parquet(
    parquet_path,
    index=False
)

In [31]:
csv_size = os.path.getsize(
    csv_path
) / (1024 ** 2)


parquet_size = os.path.getsize(
    parquet_path
) / (1024 ** 2)


print("CSV file size:",
      round(csv_size, 2),
      "MB")

print("Parquet file size:",
      round(parquet_size, 2),
      "MB")

CSV file size: 8.3 MB
Parquet file size: 2.62 MB
